# LLM Evaluation — Guide Text Generation

Tests free/local LLMs on generating identity guide sections (Core Idea + Playstyle) from structured identity data.

**Candidates:**
- HuggingFace Inference API (free tier): `mistralai/Mistral-7B-Instruct-v0.3`, `google/flan-t5-xl`
- Ollama local: Mistral 7B, Llama 3 8B, Phi-4
- Google Gemini API free tier

**Input:** Structured identity data from `docs/parsed-ids/*.md`

**Evaluation:** Qualitative grounding check — does the output match wiki data? Any hallucinated mechanics?

In [ ]:
# pip install requests
# For Ollama: install from https://ollama.com then `ollama pull mistral`

import json
import requests
from pathlib import Path
from textwrap import dedent

## 1. Load identity data and build prompts

In [ ]:
PARSED_IDS_DIR = Path("../docs/parsed-ids")

identity_data = {}
for md_file in sorted(PARSED_IDS_DIR.glob("*.md")):
    identity_data[md_file.stem] = md_file.read_text(encoding="utf-8")

print(f"Loaded {len(identity_data)} identities: {list(identity_data.keys())}")

In [ ]:
SYSTEM_PROMPT = dedent("""\
    You are a Limbus Company guide writer. Given structured identity data
    (stats, skills, passives), produce two sections:

    1. **Core Idea** (2-3 sentences): What this identity fundamentally does.
    2. **Playstyle Guide** (1 short paragraph): Key decision points, skill
       priorities, and state transitions.

    Rules:
    - ONLY reference mechanics that appear in the provided data.
    - DO NOT invent or hallucinate mechanics, skills, or numbers.
    - Use specific skill names and mechanic names from the data.
    - Be concise and actionable.
""")


def build_prompt(identity_name: str, identity_md: str) -> str:
    return f"""Below is the full structured data for the identity "{identity_name.replace('_', ' ')}".

---
{identity_md}
---

Based on the above data, write the Core Idea and Playstyle Guide."""


# Use Ring Apprentice Faust as the primary test case (most complex)
TEST_IDENTITY = "Ring_Apprentice_Faust"
test_prompt = build_prompt(TEST_IDENTITY, identity_data[TEST_IDENTITY])
print(f"Prompt length: {len(test_prompt)} chars")
print(f"\nFirst 300 chars:\n{test_prompt[:300]}")

## 2. Option A — Ollama (local)

Requires Ollama running locally (`ollama serve`) with a model pulled.

```bash
ollama pull mistral
ollama pull llama3:8b
ollama pull phi4
```

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/generate"


def query_ollama(model: str, prompt: str, system: str = SYSTEM_PROMPT) -> str | None:
    """Send a prompt to a local Ollama model and return the response."""
    try:
        resp = requests.post(OLLAMA_URL, json={
            "model": model,
            "prompt": prompt,
            "system": system,
            "stream": False,
            "options": {"temperature": 0.3, "num_predict": 512},
        }, timeout=120)
        resp.raise_for_status()
        return resp.json()["response"]
    except requests.exceptions.ConnectionError:
        print(f"  [SKIP] Ollama not running or model '{model}' not available.")
        return None
    except Exception as e:
        print(f"  [ERROR] {e}")
        return None


OLLAMA_MODELS = ["mistral", "llama3:8b", "phi4"]

ollama_results = {}
for model in OLLAMA_MODELS:
    print(f"\n{'='*60}")
    print(f"Ollama — {model}")
    print(f"{'='*60}")
    result = query_ollama(model, test_prompt)
    ollama_results[model] = result
    if result:
        print(result[:800])

## 3. Option B — HuggingFace Inference API (free tier)

Set `HF_TOKEN` environment variable or paste token below. Free tier has rate limits and model availability may vary.

In [ ]:
import os

HF_TOKEN = os.environ.get("HF_TOKEN", "")
HF_API_URL = "https://api-inference.huggingface.co/models"


def query_hf_inference(model_id: str, prompt: str) -> str | None:
    """Query HuggingFace Inference API."""
    if not HF_TOKEN:
        print("  [SKIP] HF_TOKEN not set.")
        return None
    try:
        resp = requests.post(
            f"{HF_API_URL}/{model_id}",
            headers={"Authorization": f"Bearer {HF_TOKEN}"},
            json={
                "inputs": f"{SYSTEM_PROMPT}\n\n{prompt}",
                "parameters": {"max_new_tokens": 512, "temperature": 0.3},
            },
            timeout=120,
        )
        resp.raise_for_status()
        data = resp.json()
        if isinstance(data, list) and len(data) > 0:
            return data[0].get("generated_text", str(data))
        return str(data)
    except Exception as e:
        print(f"  [ERROR] {e}")
        return None


HF_MODELS = [
    "mistralai/Mistral-7B-Instruct-v0.3",
    "google/flan-t5-xl",
]

hf_results = {}
for model_id in HF_MODELS:
    print(f"\n{'='*60}")
    print(f"HuggingFace — {model_id}")
    print(f"{'='*60}")
    result = query_hf_inference(model_id, test_prompt)
    hf_results[model_id] = result
    if result:
        print(result[:800])

## 4. Grounding check

For each generated output, check whether referenced mechanics exist in the source data.

In [ ]:
KNOWN_MECHANICS = {
    "Bleed", "Burn", "Tremor", "Rupture", "Sinking", "Poise", "Charge",
    "Bind", "Haste", "Protection", "Shield",
    "Iron Maiden", "Corpus Ingredient", "Artwork: Fascia",
    "The Self Unbound", "Flow State", "Assist Defense",
    "Unbreakable Coin", "Defense Level", "Offense Level",
    "Clash Power", "Coin Power", "Base Power", "Final Power",
    "Atk Weight", "Somatic Frisson-inspiring Melody",
}

# Mechanic terms that appear in the source identity data
source_text = identity_data[TEST_IDENTITY].lower()


def check_grounding(generated_text: str, source: str) -> dict:
    """Flag mechanic terms in generated text that don't appear in source."""
    if not generated_text:
        return {"grounded": [], "hallucinated": [], "score": 0.0}

    gen_lower = generated_text.lower()
    grounded = []
    hallucinated = []

    for mech in KNOWN_MECHANICS:
        if mech.lower() in gen_lower:
            if mech.lower() in source:
                grounded.append(mech)
            else:
                hallucinated.append(mech)

    total = len(grounded) + len(hallucinated)
    score = len(grounded) / total if total > 0 else 1.0
    return {"grounded": grounded, "hallucinated": hallucinated, "score": score}


all_outputs = {}
all_outputs.update({f"ollama/{k}": v for k, v in ollama_results.items()})
all_outputs.update({f"hf/{k}": v for k, v in hf_results.items()})

print(f"{'Model':<45s} {'Grounded':>8s} {'Halluc.':>8s} {'Score':>7s}")
print("-" * 70)
for model_name, output in all_outputs.items():
    if output is None:
        print(f"{model_name:<45s} {'N/A':>8s} {'N/A':>8s} {'N/A':>7s}")
        continue
    check = check_grounding(output, source_text)
    print(f"{model_name:<45s} {len(check['grounded']):>8d} {len(check['hallucinated']):>8d} {check['score']:>7.2f}")
    if check["hallucinated"]:
        print(f"  Hallucinated: {check['hallucinated']}")

## 5. Run on all identities (optional)

Generate guides for all available identities using the best-performing model from the evaluation above.

In [ ]:
BEST_MODEL = "mistral"  # update after running the cells above

all_guides = {}
for name, md_text in identity_data.items():
    print(f"\nGenerating guide for {name}...")
    prompt = build_prompt(name, md_text)
    result = query_ollama(BEST_MODEL, prompt)
    all_guides[name] = result
    if result:
        print(result[:400])
        print("...")

## 6. Verdict

**Selection criteria:**

| Factor | Ollama (local) | HF Inference API |
|--------|----------------|------------------|
| Latency | Low (local GPU) | Variable (queue) |
| Cost | Free | Free (rate-limited) |
| Model quality | Full model weights | Same models, hosted |
| Offline capable | Yes | No |
| Setup effort | Install Ollama + pull | API key only |

**Recommended approach for the pipeline:**
- **Primary:** Ollama with Mistral 7B or Phi-4 (fast, free, offline, MIT-licensed)
- **Fallback:** HuggingFace Inference API for CI/cloud environments
- **Upgrade path:** If quality is insufficient, try Llama 3 8B or larger models

RAG grounding (feeding structured identity JSON as context) is critical — without it, models hallucinate game mechanics.